In [33]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

In [34]:
%pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpy

In [35]:
raw_dir = Path(r"D:\ML code works\Telecom Churn Predictor\Data\raw\Employed persons by detailed occupation and age BLS")
output_dir = Path(r"D:\ML code works\Telecom Churn Predictor\Data\Integrated\Employed persons by detailed occupation and age BLS")

output_dir.mkdir(parents=True, exist_ok=True)

xlsx_files = sorted(raw_dir.glob("*.xlsx"))

print("Files found:", len(xlsx_files))
for f in xlsx_files:
    print("-", f.name)

Files found: 6
- Employed persons by detailed occupation and age 2020.xlsx
- Employed persons by detailed occupation and age 2021.xlsx
- Employed persons by detailed occupation and age 2022.xlsx
- Employed persons by detailed occupation and age 2023.xlsx
- Employed persons by detailed occupation and age 2024.xlsx
- Employed persons by detailed occupation and age 2025.xlsx


In [36]:
IT_ROLES = [
    "Computer and information systems managers",
    "Computer and information research scientists",
    "Computer systems analysts",
    "Information security analysts",
    "Computer programmers",
    "Software developers",
    "Software quality assurance analysts and testers",
    "Web developers",
    "Web and digital interface designers",
    "Computer support specialists",
    "Database administrators and architects",
    "Network and computer systems administrators",
    "Computer network architects",
    "Computer occupations, all other",
    "Computer hardware engineers",
    "Computer and mathematical occupations"
]

In [37]:
def normalize_text(x):
    if pd.isna(x):
        return np.nan
    x = str(x).replace("\n", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

In [38]:
def extract_year_from_filename(path_obj):
    match = re.search(r"(20\d{2})", path_obj.stem)
    if not match:
        raise ValueError(f"Could not extract year from filename: {path_obj.name}")
    return int(match.group(1))

In [39]:
def standardize_columns(df):
    """
    Convert different yearly column names into stable names.
    """
    df.columns = [normalize_text(col) for col in df.columns]

    # First column = occupation column
    first_col = df.columns[0]
    df = df.rename(columns={first_col: "occupation"})

    rename_map = {}

    for col in df.columns:
        c = str(col).lower()

        if col == "occupation":
            continue
        elif "total" in c and "16 years and over" in c:
            rename_map[col] = "total_employed"
        elif "16 to 19" in c:
            rename_map[col] = "age_16_19"
        elif "20 to 24" in c:
            rename_map[col] = "age_20_24"
        elif "25 to 34" in c:
            rename_map[col] = "age_25_34"
        elif "35 to 44" in c:
            rename_map[col] = "age_35_44"
        elif "45 to 54" in c:
            rename_map[col] = "age_45_54"
        elif "55 to 64" in c:
            rename_map[col] = "age_55_64"
        elif "65 years and over" in c:
            rename_map[col] = "age_65_plus"
        elif "median age" in c:
            rename_map[col] = "median_age"

    df = df.rename(columns=rename_map)
    return df

In [40]:
def load_and_filter_occupation_file(file_path):
    year = extract_year_from_filename(file_path)

    # Most BLS sheets need title rows skipped
    df = pd.read_excel(file_path, sheet_name=0, skiprows=4)

    df = standardize_columns(df)

    df["occupation"] = df["occupation"].apply(normalize_text)
    df = df[df["occupation"].notna()].copy()

    # Keep only required IT-related occupations
    df = df[df["occupation"].isin(IT_ROLES)].copy()

    # Make sure expected columns exist
    expected_cols = [
        "total_employed",
        "age_16_19",
        "age_20_24",
        "age_25_34",
        "age_35_44",
        "age_45_54",
        "age_55_64",
        "age_65_plus",
        "median_age"
    ]

    for col in expected_cols:
        if col not in df.columns:
            df[col] = np.nan

    # Convert values to numeric
    for col in expected_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["year"] = year

    final_cols = [
        "year",
        "occupation",
        "total_employed",
        "age_16_19",
        "age_20_24",
        "age_25_34",
        "age_35_44",
        "age_45_54",
        "age_55_64",
        "age_65_plus",
        "median_age"
    ]

    return df[final_cols].sort_values("occupation").reset_index(drop=True)

In [41]:
all_years = []

if len(xlsx_files) == 0:
    raise ValueError("No Excel files found. Check the raw_dir path.")

for file_path in xlsx_files:
    print(f"Processing: {file_path.name}")
    yearly_df = load_and_filter_occupation_file(file_path)
    print("Shape:", yearly_df.shape)
    all_years.append(yearly_df)

master_long_df = pd.concat(all_years, ignore_index=True)
master_long_df = master_long_df.sort_values(["occupation", "year"]).reset_index(drop=True)

print("\nFinal master_long_df shape:", master_long_df.shape)
master_long_df.head(20)

Processing: Employed persons by detailed occupation and age 2020.xlsx
Shape: (16, 11)
Processing: Employed persons by detailed occupation and age 2021.xlsx
Shape: (16, 11)
Processing: Employed persons by detailed occupation and age 2022.xlsx
Shape: (16, 11)
Processing: Employed persons by detailed occupation and age 2023.xlsx
Shape: (16, 11)
Processing: Employed persons by detailed occupation and age 2024.xlsx
Shape: (16, 11)
Processing: Employed persons by detailed occupation and age 2025.xlsx
Shape: (16, 11)

Final master_long_df shape: (96, 11)


,year,occupation,total_employed,age_16_19,age_20_24,age_25_34,age_35_44,age_45_54,age_55_64,age_65_plus,median_age
0,2020,Computer and information research scientists,42.0,0.0,3.0,14.0,9.0,9.0,5.0,2.0,NaN
1,2021,Computer and information research scientists,41.0,0.0,2.0,20.0,9.0,4.0,4.0,1.0,NaN
2,2022,Computer and information research scientists,41.0,0.0,3.0,17.0,3.0,7.0,10.0,0.0,NaN
3,2023,Computer and information research scientists,44.0,0.0,2.0,29.0,7.0,3.0,2.0,1.0,NaN
4,2024,Computer and information research scientists,36.0,1.0,8.0,9.0,9.0,6.0,3.0,0.0,NaN
5,2025,Computer and information research scientists,29.0,0.0,0.0,13.0,8.0,5.0,2.0,1.0,NaN
6,2020,Computer and information systems managers,744.0,0.0,7.0,128.0,222.0,226.0,138.0,23.0,45.4
7,2021,Computer and information systems managers,715.0,2.0,9.0,113.0,209.0,212.0,149.0,21.0,45.7
8,2022,Computer and information systems managers,764.0,2.0,10.0,132.0,225.0,222.0,141.0,31.0,45.7
9,2023,Computer and information systems managers,793.0,0.0,8.0,156.0,243.0,223.0,139.0,24.0,44.8


In [42]:
wide_total_df = master_long_df.pivot(
    index="occupation",
    columns="year",
    values="total_employed"
).reset_index()

wide_total_df.columns = ["occupation"] + [f"total_employed_{col}" for col in wide_total_df.columns[1:]]
wide_total_df = wide_total_df.sort_values("occupation").reset_index(drop=True)

print(wide_total_df.shape)
wide_total_df.head()

(16, 7)


,occupation,total_employed_2020,total_employed_2021,total_employed_2022,total_employed_2023,total_employed_2024,total_employed_2025
0,Computer and information research scientists,42.0,41.0,41.0,44.0,36.0,29.0
1,Computer and information systems managers,744.0,715.0,764.0,793.0,732.0,646.0
2,Computer and mathematical occupations,5603.0,5688.0,6171.0,6502.0,6386.0,6711.0
3,Computer hardware engineers,89.0,88.0,90.0,71.0,85.0,94.0
4,Computer network architects,107.0,102.0,98.0,104.0,139.0,96.0


In [43]:
wide_all_df = master_long_df.pivot(index="occupation", columns="year")

wide_all_df.columns = [f"{metric}_{year}" for metric, year in wide_all_df.columns]
wide_all_df = wide_all_df.reset_index()
wide_all_df = wide_all_df.sort_values("occupation").reset_index(drop=True)

print(wide_all_df.shape)
wide_all_df.head()

(16, 55)


,occupation,total_employed_2020,total_employed_2021,total_employed_2022,total_employed_2023,total_employed_2024,total_employed_2025,age_16_19_2020,age_16_19_2021,age_16_19_2022,...,age_65_plus_2022,age_65_plus_2023,age_65_plus_2024,age_65_plus_2025,median_age_2020,median_age_2021,median_age_2022,median_age_2023,median_age_2024,median_age_2025
0,Computer and information research scientists,42.0,41.0,41.0,44.0,36.0,29.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
1,Computer and information systems managers,744.0,715.0,764.0,793.0,732.0,646.0,0.0,2.0,2.0,...,31.0,24.0,31.0,21.0,45.4,45.7,45.7,44.8,47.0,46.4
2,Computer and mathematical occupations,5603.0,5688.0,6171.0,6502.0,6386.0,6711.0,23.0,28.0,32.0,...,216.0,218.0,212.0,214.0,40.6,40.7,40.6,40.8,40.7,40.7
3,Computer hardware engineers,89.0,88.0,90.0,71.0,85.0,94.0,0.0,0.0,0.0,...,2.0,1.0,4.0,1.0,42.7,38.2,42.5,42.8,42.7,42.2
4,Computer network architects,107.0,102.0,98.0,104.0,139.0,96.0,0.0,0.0,0.0,...,4.0,1.0,6.0,4.0,43.8,45.0,43.2,43.9,45.1,45.1


In [44]:
master_long_path = output_dir / "it_occupations_master_long.csv"
wide_total_path = output_dir / "it_occupations_wide_total.csv"
wide_all_path = output_dir / "it_occupations_wide_all_metrics.csv"
excel_path = output_dir / "it_occupations_merged.xlsx"

master_long_df.to_csv(master_long_path, index=False)
wide_total_df.to_csv(wide_total_path, index=False)
wide_all_df.to_csv(wide_all_path, index=False)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    master_long_df.to_excel(writer, sheet_name="master_long", index=False)
    wide_total_df.to_excel(writer, sheet_name="wide_total", index=False)
    wide_all_df.to_excel(writer, sheet_name="wide_all_metrics", index=False)

print("Saved files:")
print(master_long_path)
print(wide_total_path)
print(wide_all_path)
print(excel_path)

Saved files:
D:\ML code works\Telecom Churn Predictor\Data\Integrated\Employed persons by detailed occupation and age BLS\it_occupations_master_long.csv
D:\ML code works\Telecom Churn Predictor\Data\Integrated\Employed persons by detailed occupation and age BLS\it_occupations_wide_total.csv
D:\ML code works\Telecom Churn Predictor\Data\Integrated\Employed persons by detailed occupation and age BLS\it_occupations_wide_all_metrics.csv
D:\ML code works\Telecom Churn Predictor\Data\Integrated\Employed persons by detailed occupation and age BLS\it_occupations_merged.xlsx


In [45]:
print("Unique occupations:", master_long_df["occupation"].nunique())
print("Years:", sorted(master_long_df["year"].unique()))
print(master_long_df.groupby("year")["occupation"].count())

Unique occupations: 16
Years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
year
2020    16
2021    16
2022    16
2023    16
2024    16
2025    16
Name: occupation, dtype: int64


In [46]:
master_long_df[master_long_df["occupation"] == "Software developers"]

,year,occupation,total_employed,age_16_19,age_20_24,age_25_34,age_35_44,age_45_54,age_55_64,age_65_plus,median_age
72,2020,Software developers,1883.0,4.0,109.0,675.0,529.0,303.0,220.0,43.0,39.1
73,2021,Software developers,1932.0,4.0,122.0,700.0,523.0,338.0,205.0,40.0,38.6
74,2022,Software developers,2085.0,0.0,133.0,746.0,553.0,376.0,234.0,43.0,38.9
75,2023,Software developers,2134.0,7.0,132.0,779.0,582.0,349.0,220.0,65.0,38.7
76,2024,Software developers,2077.0,3.0,129.0,742.0,567.0,366.0,230.0,40.0,38.8
77,2025,Software developers,2254.0,3.0,140.0,808.0,655.0,380.0,218.0,51.0,38.6
